In [ ]:
!pip install bagpy --quiet
!pip install pyarrow --quiet
!pip install zstandard --quiet

In [ ]:
import re
from pathlib import Path
import tempfile, shutil
import numpy as np, pandas as pd
from bagpy import bagreader
import zstandard as zstd

def extract_values(s: str) -> np.ndarray:
    pat = r'[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+[eE][-+]?\d+|[-+]?\d+'
    matches = re.findall(pat, str(s))
    return np.array([float(x) for x in re.findall(pat, str(s))], dtype=float)

def decompress_zst(p: Path) -> tuple[Path, Path | None]:
    if p.suffix != ".zst":
        return p, None
    tmpdir = Path(tempfile.mkdtemp(prefix="bagzst_"))
    out = tmpdir / p.name[:-4]  # quita .zst
    print(f"Decompressing {p} to {out} ...")
    
    dctx = zstd.ZstdDecompressor()
    with p.open("rb") as fin, out.open("wb") as fout:
        dctx.copy_stream(fin, fout)
        
    return out, tmpdir
# ================== Paths ==================
bag_paths = [
    #"./data/dataset_path"
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path",
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"./data/dataset_path"
    #"/nexus/Downloads/2025-11-19-20-13-11.bag"
    #new sim bags
   "./data/dataset_path",
   "./data/dataset_path",
   "./data/dataset_path",
   "./data/dataset_path",
   "./data/dataset_path",
   "./data/dataset_path"
   
    
    
    

    
]

outdir = Path("./data/dataset_path")
outdir.mkdir(parents=True, exist_ok=True)

for raw_path in bag_paths:
    src = Path(raw_path)
    
    if not src.exists():
        print(f"[WARN] No existe: {src}")
        continue
    
    try: 
        bag_file_path, tmpdir = decompress_zst(src)
    except Exception as e:
        print(f"[ERROR] Decompressing {src}: {e}")
        continue
    #bag_file, tmpdir = decompress_zst(src)
    #print(f"\nProcessing {bag_file} ...")

    try:
        print(f"\nProcessing: {bag_file_path.name}")
        
        b = bagreader(str(bag_file_path)) # here i passed the list instead of the current file 
        # -----------------Processing  decompressed bag ----------------------------------
        state_csv = b.message_by_topic("/estimated_robot_state")
        imu_csv = b.message_by_topic("/gazebo/imu")
        df_state = None

        if state_csv and Path(state_csv).exists():
            df_state = pd.read_csv(state_csv)
            for col in ("ee_pos", "ee_vel", "ee_forces"):
                if col in df_state.columns:
                    df_state[col] = df_state[col].apply(extract_values)
        
        df_imu = None
        if imu_csv and Path(imu_csv).exists():
            df_imu = pd.read_csv(imu_csv).rename(columns={
                "linear_acceleration.x": "ax",
                "linear_acceleration.y": "ay",
                "linear_acceleration.z": "az",
                "angular_velocity.x": "wx",
                "angular_velocity.y": "wy",
                "angular_velocity.z": "wz",
            })
        
        #-----------------------Generating clean stem name -----------------------------
        stem = bag_file_path.stem
        for suf in (".bag", ".bz2", ".lz4"):  
            if stem.endswith(suf):
                stem = stem[: -len(suf)]

        #----------------------- Saving to parquet --------------------------------------
        if df_state is not None and not df_state.empty:
            state_parquet = outdir / f"{stem}_state.parquet"
            df_state.to_parquet(state_parquet, compression="zstd")
            print(f" Saved state → {state_parquet.name}")
        else:
            print(" (no state data)")

        if df_imu is not None and not df_imu.empty:
            imu_parquet = outdir / f"{stem}_imu.parquet"
            df_imu.to_parquet(imu_parquet, compression="zstd")
            print(f" Saved IMU   → {imu_parquet.name}")
        else:
            print(" (no IMU data)")

    except Exception as e:
        print(f"[ERROR] Failed to process {bag_file_path.name}: {e}")
        import traceback
        traceback.print_exc()

    finally:
        if 'bag_file_path' in locals() and bag_file_path != src and bag_file_path.exists():
            bag_file_path.unlink()
            

In [ ]:
# Decompress .zst bag (overwrite existing .bag automatically)
#!zstd -d -f ./data/dataset_path -o ./data/dataset_path

#--------------------------- Reindex the bag - 
!rosbag reindex ./data/dataset_path

In [ ]:
!zstd -f -19 ./data/dataset_path -o ./data/dataset_path

In [ ]:
##chosen intuitive noise addition
import os
import pandas as pd
import numpy as np
import json

def generate_ou_colored_noise(size, dt=1/200, sigma=0.2, tau=0.7, seed=42):
    np.random.seed(seed)
    n_steps, n_dims = size
    noise = np.zeros((n_steps, n_dims))
    for d in range(n_dims):
        for t in range(1, n_steps):
            noise[t, d] = noise[t-1, d] + (-(1/tau) * noise[t-1, d]) * dt + sigma * np.sqrt(2 / tau) * np.sqrt(dt) * np.random.randn()
    return noise

# Low pass filter helper function 
def generate_lowpass_noise(shape, std=0.1, cutoff = 15, fs = 200, order = 2):
    from scipy.signal import butter, filtfilt
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    noise = np.random.normal(0, std, shape)
    filtered_noise = filtfilt(b, a, noise, axis=0)
    return filtered_noise

def apply_imu_noise(acc, omega, file_path=None, dt=1/100):
    np.random.seed(52)

    # ----- White Noise for Acceleration -----
    # Adjust std for each axis
    acc_white_std = np.array([0.5, 0.80, 1.20])  # x, y, z (↑ y noise, ↓ z noise)
    white_noise_acc = np.random.normal(0, acc_white_std, acc.shape)
    low_pass_filtered_noise  = generate_lowpass_noise(acc.shape, std=0.1, cutoff=25, fs=(int(1/dt)), order=2)
    acc_bias = np.random.normal([-0.206e-06, 0.022e-07, 3.721e-05])  #x,y,z axis stds
    acc_noisy = acc + white_noise_acc + 0.2 * low_pass_filtered_noise + acc_bias

    # ----- Colored Noise for Angular Velocities (gyro w/ ou)-----
    # Axis-specific tuning (X, Y, Z)
   # sigma_vals = np.array([0.35, 0.3, 0.069])  # ↓ sigma for X/Y, keep Z close
    sigma_vals = np.array([0.15, 0.13, 0.05]) 
   # tau_vals = np.array([0.15, 0.08, 0.13])         # ↓ tau for spikiness in X/Y
    tau_vals   = np.array([0.20, 0.15, 0.12]) 

    colored_noise_omega = np.zeros_like(omega)
    for i in range(3):
        colored_noise = generate_ou_colored_noise(
            (omega.shape[0], 1), dt=dt, sigma=sigma_vals[i], tau=tau_vals[i], seed=52+i
        )
        colored_noise_omega[:, i] = colored_noise.flatten()
        #omega_bias= np.array([4.6576e-05, -0.01136, 0.100102]) 
        omega_bias = np.array([0.0, 0.0, 0.03])

        
    omega_noisy = omega + colored_noise_omega + np.random.normal(0, 0.002, omega.shape) + omega_bias
    
    #gyro_white_std = np.array([0.03, 0.011, 0.0045])
    gyro_white_std = np.array([0.008, 0.006, 0.003])


    # ----- Clip values -----  
    acc_noisy = np.clip(acc_noisy, -25, 25)
    
    omega_noisy = np.clip(omega_noisy, -25, 25)

    # ----- Save log -----
    if file_path is not None:
        log_data = {
            "acc_white_std": acc_white_std.tolist(),
            "omega_colored_sigma": sigma_vals.tolist(),
            "acc_bias": acc_bias.tolist(),
            "acc_lowpass_weight": 0.2,
            "gyro_white_std": gyro_white_std.tolist(),
            "omega_colored_tau": tau_vals.tolist(),
            "gyro_bias": omega_bias.tolist(),
            "sampling_dt": dt,
            "colored_noise_type": "Ornstein-Uhlenbeck"
        }
        
        log_dir = "./data/dataset_path"
        os.makedirs(log_dir, exist_ok=True)
        log_file = os.path.join(log_dir, f"{os.path.basename(file_path)}_noise.json")
        with open(log_file, "w") as f:
            json.dump(log_data, f, indent=2)
    
    return acc_noisy, omega_noisy

# File path
#file_path = "./data/dataset_path"
#file_path = "./data/dataset_path",
#file_path = "./data/dataset_path",
#file_path = "./data/dataset_path",
#file_path = "./data/dataset_path",
#file_path = "./data/dataset_path",
#file_path = "./data/dataset_path",
#file_path = "./data/dataset_path",
#file_path = "./data/dataset_path",
#file_path = "./data/dataset_path"
#file_path = "./data/dataset_path"
#file_path = "home/nexus/Desktop/simdata/SIMDATA_ROSBAGS/concreteskip-02_imu.csv"
#file_path = "./data/dataset_path"
#file_path = "./data/dataset_path"
#file_path = "./data/dataset_path"
file_path = "./data/dataset_path"

# Load CSV
df = pd.read_csv(file_path)

# Extract IMU
acc = df[['linear_acceleration.x', 'linear_acceleration.y', 'linear_acceleration.z']].to_numpy()
omega = df[['angular_velocity.x', 'angular_velocity.y', 'angular_velocity.z']].to_numpy()

# Inject noise
acc_noisy, omega_noisy = apply_imu_noise(acc, omega, file_path=file_path)

# Replace in DataFrame
df[['linear_acceleration.x', 'linear_acceleration.y', 'linear_acceleration.z']] = acc_noisy
df[['angular_velocity.x', 'angular_velocity.y', 'angular_velocity.z']] = omega_noisy

# Save
save_path = file_path.replace("_imu.csv", "_noiseimu.csv")
df.to_csv(save_path, index=False)

print(f"Noisy IMU data saved to: {save_path}")


In [ ]:
#!/usr/bin/env python3
import os
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial.transform import Rotation as R

# ----------------------------------------------------------
# CONFIG — YOUR TARGET FOLDERS (ONLY THESE WILL BE USED)
# ----------------------------------------------------------
TARGET_BASE = [
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path",
    "./data/dataset_path"
]

OUTPUT_DIR  = Path("./data/dataset_path")
OUT_BODYCSV = Path("./data/dataset_path")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_BODYCSV.mkdir(parents=True, exist_ok=True)

LEGS = ['LF', 'RF', 'LH', 'RH']

# ----------------------------------------------------------
# Utility functions
# ----------------------------------------------------------
def parse_vec12_series(series: pd.Series) -> np.ndarray:
    n = len(series)
    out = np.full((n, 12), np.nan)
    for i, s in enumerate(series.values):
        try:
            if isinstance(s, (list, tuple, np.ndarray)):
                v = np.asarray(s, float)
            else:
                v = np.fromstring(str(s).strip().strip("[]").replace(",", " "), sep=" ")
            if v.size:
                out[i, :min(12, v.size)] = v[:12]
        except:
            pass
    return out

def parse_list_series(series: pd.Series, expected_len=None):
    arrs=[]
    for s in series.values:
        try:
            if isinstance(s,(list,tuple,np.ndarray)):
                v=np.asarray(s,float)
            else:
                v=np.array(ast.literal_eval(str(s)), float)
        except:
            v=np.array([], float)
        arrs.append(v)
    maxlen = max((len(v) for v in arrs), default=0)
    if expected_len: maxlen=max(expected_len,maxlen)
    out=np.full((len(arrs), maxlen), np.nan)
    for i,v in enumerate(arrs):
        out[i,:min(len(v),maxlen)] = v[:maxlen]
    return out

def w2b_points(Pw, Rwb, p_wb):
    return np.einsum('nij,nj->ni', Rwb.transpose(0,2,1), Pw - p_wb)

def w2b_vecs(Vw, Rwb):
    return np.einsum('nij,nj->ni', Rwb.transpose(0,2,1), Vw)

def split_legs(arr12):
    return {
        'LF': arr12[:,0:3],
        'RF': arr12[:,3:6],
        'LH': arr12[:,6:9],
        'RH': arr12[:,9:12]
    }

# ----------------------------------------------------------
# PROCESS ONLY THE TARGET DATASETS
# ----------------------------------------------------------
for folder in TARGET_BASE:

    folder_path = Path(folder)
    name = folder_path.name

    state_csv = folder_path / "estimated_robot_state.csv"
    imu_csv   = folder_path / "gazebo-imu.csv"

    print("\n=====================================================")
    print(f" PROCESSING: {name}")
    print("=====================================================\n")

    if not state_csv.exists():
        print(" MISSING estimated_robot_state.csv — SKIPPING")
        continue

    # -------- Load CSVs --------
    state_df = pd.read_csv(state_csv)
    imu_df   = pd.read_csv(imu_csv) if imu_csv.exists() else pd.DataFrame()

    # -------- Parse world values --------
    pos12_w = parse_vec12_series(state_df["ee_pos"])
    vel12_w = parse_vec12_series(state_df["ee_vel"]) if "ee_vel" in state_df else np.zeros_like(pos12_w)

    q  = parse_list_series(state_df["joint_state.position"], expected_len=12)
    qd = parse_list_series(state_df["joint_state.velocity"], expected_len=12)

    # Contacts
    cb=[]
    for c in state_df["ee_contact"].values:
        try:
            if isinstance(c,(list,tuple,np.ndarray)):
                v=np.asarray(c,int)
            else:
                v=np.array(ast.literal_eval(str(c)),int)
        except:
            v=np.array([0,0,0,0])
        cb.append(v[:4])
    contacts_bool = np.vstack(cb)

    # Trim
    N = min(len(state_df), len(pos12_w), len(vel12_w), len(q), len(qd), len(contacts_bool))
    pos12_w, vel12_w, q, qd, contacts_bool = (
        pos12_w[:N], vel12_w[:N], q[:N], qd[:N], contacts_bool[:N]
    )
    sdf = state_df.iloc[:N].reset_index(drop=True)

    # ----------------------------------------------------------
    # World → Body transform
    # ----------------------------------------------------------
    need = [
        "base_pose.position.x","base_pose.position.y","base_pose.position.z",
        "base_pose.orientation.x","base_pose.orientation.y","base_pose.orientation.z","base_pose.orientation.w"
    ]
    missing = [c for c in need if c not in sdf.columns]

    if missing:
        print(" ⚠ Missing base pose, keeping world coords as body coords")
        pos_b = split_legs(pos12_w)
        vel_b = split_legs(vel12_w)
    else:
        base_pos  = sdf[["base_pose.position.x","base_pose.position.y","base_pose.position.z"]].to_numpy(float)
        base_quat = sdf[["base_pose.orientation.x","base_pose.orientation.y","base_pose.orientation.z","base_pose.orientation.w"]].to_numpy(float)

        n = np.linalg.norm(base_quat,axis=1)
        n[n==0]=1
        base_quat /= n[:,None]

        Rwbs = np.array([R.from_quat(qi).as_matrix() for qi in base_quat])

        pos_w = split_legs(pos12_w)
        vel_w = split_legs(vel12_w)

        pos_b = {leg: w2b_points(pos_w[leg],Rwbs,base_pos) for leg in LEGS}
        vel_b = {leg: w2b_vecs(vel_w[leg],Rwbs) for leg in LEGS}

    # -------- Write debug CSV --------
    out_csv = OUT_BODYCSV / f"{name}_body_corrected.csv"
    bd = sdf.copy()

    for leg in LEGS:
        for idx, ax in enumerate(['x','y','z']):
            bd[f"ee_pos.{leg}.body.{ax}"] = pos_b[leg][:,idx]
            bd[f"ee_vel.{leg}.body.{ax}"] = vel_b[leg][:,idx]

    bd.to_csv(out_csv, index=False)
    print(" ✔ Wrote body CSV:", out_csv)

    # ----------------------------------------------------------
    # Build feature matrices
    # ----------------------------------------------------------
    def collect(prefix, df):
        mats=[]
        for leg in LEGS:
            for ax in ['x','y','z']:
                col=f"{prefix}.{leg}.body.{ax}"
                mats.append(pd.to_numeric(df[col],errors="coerce").to_numpy())
        return np.stack(mats,axis=1)

    p_body = collect("ee_pos", bd)
    v_body = collect("ee_vel", bd)

    # Final trim
    N = min(len(bd), len(q), len(qd))
    q, qd, p_body, v_body = q[:N], qd[:N], p_body[:N], v_body[:N]
    contacts_trim = contacts_bool[:N]

    # Joint reorder
    order=[6,7,8, 0,1,2, 9,10,11, 3,4,5]
    if q.shape[1] >= 12:  q = q[:,order]
    if qd.shape[1] >= 12: qd = qd[:,order]
    p_body = p_body[:,order]
    v_body = v_body[:,order]

    # Labels
    reorder = contacts_trim[:, [2,0,3,1]]
    labels = np.dot(reorder,[8,4,2,1]).astype(int)

    # ----------------------------------------------------------
    # SAVE OUTPUT
    # ----------------------------------------------------------
    terrain_dir = OUTPUT_DIR / name
    terrain_dir.mkdir(parents=True, exist_ok=True)

    np.save(terrain_dir/"q.npy", q)
    np.save(terrain_dir/"qd.npy", qd)
    np.save(terrain_dir/"p.npy", p_body)
    np.save(terrain_dir/"v.npy", v_body)
    np.save(terrain_dir/"contacts.npy", contacts_trim)
    np.save(terrain_dir/"contacts_labels.npy", labels)
    np.save(terrain_dir/"preprocessed_data.npy",
            np.concatenate((q,qd,p_body,v_body),axis=1))

    print(f"\n ✔ Saved → {terrain_dir}")
    print(" -----------------------------------------------------")


In [ ]:
#!/usr/bin/env python3
import os
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial.transform import Rotation as R

# ----------------------------------------------------------
# FIND MATCHING PARQUET DATASETS
# ----------------------------------------------------------
DATA_ROOT   = Path("./data/dataset_path")
OUTPUT_DIR  = Path("./data/dataset_path")
OUT_BODYCSV = Path("./data/dataset_path")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_BODYCSV.mkdir(parents=True, exist_ok=True)

LEGS = ['LF', 'RF', 'LH', 'RH']

# ----------------------------------------------------------
# PROCESS ONLY THESE DATASET NAMES
# ----------------------------------------------------------
DATASETS_TO_USE = [
    "2025-11-27-00-10-40",
    "2025-11-27-23-36-32",
    "2025-11-27-23-42-25",
    "2025-11-27-23-46-54",
    "2025-11-28-00-12-53",
    "2025-11-28-00-46-21"
]

# ----------------------------------------------------------
# Utility functions (unchanged)
# ----------------------------------------------------------
def parse_vec12_series(series: pd.Series) -> np.ndarray:
    n = len(series)
    out = np.full((n, 12), np.nan)
    for i, s in enumerate(series.values):
        try:
            if isinstance(s, (list, tuple, np.ndarray)):
                v = np.asarray(s, float)
            else:
                v = np.fromstring(str(s).strip().strip("[]").replace(",", " "), sep=" ")
            if v.size:
                out[i, :min(12, v.size)] = v[:12]
        except:
            pass
    return out

def parse_list_series(series: pd.Series, expected_len=None):
    arrs=[]
    for s in series.values:
        try:
            if isinstance(s,(list,tuple,np.ndarray)):
                v=np.asarray(s,float)
            else:
                v=np.array(ast.literal_eval(str(s)), float)
        except:
            v=np.array([], float)
        arrs.append(v)
    maxlen = max((len(v) for v in arrs), default=0)
    if expected_len: maxlen=max(expected_len,maxlen)
    out=np.full((len(arrs), maxlen), np.nan)
    for i,v in enumerate(arrs):
        out[i,:min(len(v),maxlen)] = v[:maxlen]
    return out

def w2b_points(Pw, Rwb, p_wb):
    return np.einsum('nij,nj->ni', Rwb.transpose(0,2,1), Pw - p_wb)

def w2b_vecs(Vw, Rwb):
    return np.einsum('nij,nj->ni', Rwb.transpose(0,2,1), Vw)

def split_legs(arr12):
    return {
        'LF': arr12[:,0:3],
        'RF': arr12[:,3:6],
        'LH': arr12[:,6:9],
        'RH': arr12[:,9:12]
    }

# ----------------------------------------------------------
# FIND DATASETS (CSV version)
# ----------------------------------------------------------
state_files = []
for base in DATASETS_TO_USE:
    folder = DATA_ROOT / base
    state_csv = folder / "estimated_robot_state.csv"
    imu_csv   = folder / "gazebo-imu.csv"

    if state_csv.exists():
        state_files.append((base, state_csv, imu_csv))
    else:
        print(f"[WARNING] Missing CSV file: {state_csv}")

print(f"\nWill process {len(state_files)} datasets.\n")

        
for TARGET_BASE, state_path, imu_path in state_files:

    print("\n=====================================================")
    print(f" PROCESSING: {TARGET_BASE}")
    print("=====================================================\n")

    # -------- Read CSV instead of parquet --------
    state_df = pd.read_csv(state_path)
    imu_df = pd.read_csv(imu_path) if imu_path.exists() else pd.DataFrame()


    # -------- Read parquet --------
    #state_df = pd.read_parquet(state_path)
    #imu_df = pd.read_parquet(imu_path) if imu_path.exists() else pd.DataFrame()

    # -------- Parse world values --------
    pos12_w = parse_vec12_series(state_df["ee_pos"])
    vel12_w = parse_vec12_series(state_df["ee_vel"]) if "ee_vel" in state_df else np.zeros_like(pos12_w)

    q  = parse_list_series(state_df["joint_state.position"], expected_len=12)
    qd = parse_list_series(state_df["joint_state.velocity"], expected_len=12)

    # Contacts
    cb=[]
    for c in state_df["ee_contact"].values:
        try:
            if isinstance(c,(list,tuple,np.ndarray)):
                v=np.asarray(c,int)
            else:
                v=np.array(ast.literal_eval(str(c)),int)
        except:
            v=np.array([0,0,0,0])
        cb.append(v[:4])
    contacts_bool = np.vstack(cb)

    # Trim
    N = min(len(state_df), len(pos12_w), len(vel12_w), len(q), len(qd), len(contacts_bool))
    pos12_w, vel12_w, q, qd, contacts_bool = (
        pos12_w[:N], vel12_w[:N], q[:N], qd[:N], contacts_bool[:N]
    )
    sdf = state_df.iloc[:N].reset_index(drop=True)

    # -------- World → Body --------
    need = [
        "base_pose.position.x","base_pose.position.y","base_pose.position.z",
        "base_pose.orientation.x","base_pose.orientation.y","base_pose.orientation.z","base_pose.orientation.w"
    ]
    missing = [c for c in need if c not in sdf.columns]

    if missing:
        pos_b = split_legs(pos12_w)
        vel_b = split_legs(vel12_w)
    else:
        base_pos  = sdf[["base_pose.position.x","base_pose.position.y","base_pose.position.z"]].to_numpy(float)
        base_quat = sdf[["base_pose.orientation.x","base_pose.orientation.y","base_pose.orientation.z","base_pose.orientation.w"]].to_numpy(float)

        n = np.linalg.norm(base_quat,axis=1); n[n==0]=1
        base_quat /= n[:,None]

        Rwbs = np.array([R.from_quat(qi).as_matrix() for qi in base_quat])

        pos_w = split_legs(pos12_w)
        vel_w = split_legs(vel12_w)

        pos_b = {leg: w2b_points(pos_w[leg],Rwbs,base_pos) for leg in LEGS}
        vel_b = {leg: w2b_vecs(vel_w[leg],Rwbs) for leg in LEGS}

    # -------- Save debug CSV --------
    out_csv = OUT_BODYCSV / f"{TARGET_BASE}_body_corrected.csv"
    bd = sdf.copy()

    for leg in LEGS:
        for ax in ['x','y','z']:
            bd[f"ee_pos.{leg}.body.{ax}"] = pos_b[leg][:, ['x','y','z'].index(ax)]
            bd[f"ee_vel.{leg}.body.{ax}"] = vel_b[leg][:, ['x','y','z'].index(ax)]

    bd.to_csv(out_csv, index=False)
    print(" ✔ Wrote body CSV:", out_csv)

    # -------- Build model matrices --------
    def collect(prefix, df):
        mats=[]
        for leg in LEGS:
            for ax in ['x','y','z']:
                col=f"{prefix}.{leg}.body.{ax}"
                mats.append(pd.to_numeric(df[col],errors="coerce").to_numpy())
        return np.stack(mats,axis=1)

    p_body = collect("ee_pos", bd)
    v_body = collect("ee_vel", bd)

    # Final trim
    N = min(len(bd), len(q), len(qd))
    q, qd, p_body, v_body = q[:N], qd[:N], p_body[:N], v_body[:N]
    contacts_trim = contacts_bool[:N]

    # Joint reorder
    order=[6,7,8, 0,1,2, 9,10,11, 3,4,5]
    if q.shape[1] >= 12:  q = q[:,order]
    if qd.shape[1] >= 12: qd = qd[:,order]
    p_body = p_body[:,order]
    v_body = v_body[:,order]

    # Labels
    reorder = contacts_trim[:, [2,0,3,1]]
    labels = np.dot(reorder,[8,4,2,1]).astype(int)

    # -------- SAVE --------
    terrain_dir = OUTPUT_DIR / TARGET_BASE
    terrain_dir.mkdir(parents=True, exist_ok=True)

    np.save(terrain_dir/"q.npy", q)
    np.save(terrain_dir/"qd.npy", qd)
    np.save(terrain_dir/"p.npy", p_body)
    np.save(terrain_dir/"v.npy", v_body)
    np.save(terrain_dir/"contacts.npy", contacts_trim)
    np.save(terrain_dir/"contacts_labels.npy", labels)
    np.save(terrain_dir/"preprocessed_data.npy", np.concatenate((q,qd,p_body,v_body),axis=1))

    print(f"\n ✔ Saved → {terrain_dir}")
    print(" -----------------------------------------------------")


In [ ]:
#!/usr/bin/env python3
import numpy as np
from pathlib import Path

def repair_nans(arr):
    """Fill NaNs by linear interpolation along time, column-wise."""
    arr = arr.copy()
    for i in range(arr.shape[1]):
        col = arr[:, i]
        nans = np.isnan(col)
        if np.any(nans):
            not_nan = ~nans
            # interpolate values for NaNs
            col[nans] = np.interp(np.flatnonzero(nans),
                                  np.flatnonzero(not_nan),
                                  col[not_nan])
        arr[:, i] = col
    return arr


preprocessed_dir = Path("./data/dataset_path")
final_output     = Path("./data/dataset_path")
final_output.mkdir(parents=True, exist_ok=True)

terrain_names = [
    "2025-11-27-00-10-40",
    "2025-11-27-23-36-32",
    "2025-11-27-23-42-25",
    "2025-11-27-23-46-54",
    "2025-11-28-00-12-53",
    "2025-11-28-00-46-21"
]

for name in terrain_names:

    print("\n===========================================")
    print(f"Processing: {name}")
    print("===========================================")

    terrain_path = preprocessed_dir / name
    if not terrain_path.exists():
        print(f" Missing folder: {terrain_path}, skipping.")
        continue

    # Load files
    try:
        rq  = np.load(terrain_path / "q.npy")
        rqd = np.load(terrain_path / "qd.npy")
        rp  = np.load(terrain_path / "p.npy")
        rv  = np.load(terrain_path / "v.npy")
        

    except FileNotFoundError as e:
        print(f" Missing file inside {name}: {e}")
        continue

    labels_path = terrain_path / "contacts_labels.npy"
    labels = np.load(labels_path) if labels_path.exists() else np.zeros(rq.shape[0], dtype=np.int32)

    # Add noise
    rq  += np.random.normal(0, 0.005, rq.shape)
    rqd += np.random.normal(0, 0.01, rqd.shape)
    rp  += np.random.normal(0, 0.001, rp.shape)
    rv  += np.random.normal(0, 0.002, rv.shape)

    # Align lengths
    N = min(rq.shape[0], rqd.shape[0], rp.shape[0], rv.shape[0], labels.shape[0])
    rq, rqd, rp, rv, labels = rq[:N], rqd[:N], rp[:N], rv[:N], labels[:N]

    # Build final tensor
    data = np.concatenate((rq, rqd, rp, rv), axis=1).astype(np.float32)

    # Save output
    out_dir = final_output / name
    out_dir.mkdir(parents=True, exist_ok=True)

    np.save(out_dir / "data_reordered.npy", data)
    np.save(out_dir / "labels_reordered.npy", labels.astype(np.int32))

    print(f"✔ Saved: {out_dir}")
    print(f"  data shape: {data.shape}")
    print(f"  labels shape: {labels.shape}")
    print(f"  size: {data.nbytes/(1024*1024):.2f} MB")


In [ ]:
import os
import numpy as np
import collections

# Directory to save all split files
save_dir = "./data/dataset_path"
os.makedirs(save_dir, exist_ok=True)

def print_time_summary_for_terrains(terrains, time_idx=-1):
    for name, (data, labels) in terrains.items():
        print(f"\n=== {name} ===")
        
        if not isinstance(data, np.ndarray):
            print(f"  Warning: data for {name} is not a numpy array ({type(data)})")
            continue
        
        if data.ndim == 2:
            if time_idx >= data.shape[1] or time_idx < -data.shape[1]:
                print(f"  Invalid time_idx {time_idx} for shape {data.shape}")
            else:
                time_col = data[:, time_idx]
                print(f"  Time range: {time_col.min():.3f} → {time_col.max():.3f}")
        else:
            print(f"  Unexpected data shape: {data.shape}")

# Define all terrains to process. NOTE: Fixed the WAREHOUSEASPHALTTROT entry.
terrains = { #trot only terrains
    "2025-11-27-00-10-40": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),
    
    "2025-11-27-23-36-32": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),
    "2025-11-27-23-42-25": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),
    
    "2025-11-27-23-46-54": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),
    
    "2025-11-28-00-12-53": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),
    
    "2025-11-28-00-46-21": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),
    "2025-11-21-16-14-15": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),

    "2025-11-21-19-58-18": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),

    "2025-11-21-20-18-51": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),

    "2025-11-22-01-30-44": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),

    "2025-11-24-02-10-41": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),

    "2025-11-25-22-26-08": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),

    "2025-11-26-01-01-29": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),

    "2025-11-26-20-34-54": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")
    ),

  #   np.load("./data/dataset_path"),
      #  np.load("./data/dataset_path")
    #), # the new terrains i added end upto here only, the ones below i have a trained model that uses them.... 

    "concretetrot0.9": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "2025-10-12-06-30-58mudflyingtrot": (
        np.load("./data/dataset_path"),        
        np.load("./data/dataset_path")),
 
    "2025-10-12-05-44-08sandpaperftterrain": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
 
    "STANDING_TROT": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
 
    "SANDPAPER": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
 
    "MIDDLE_PEBBLE": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
 
# "LINDYHOP_ROAD": (
#      np.load("./data/dataset_path"),
#      np.load("./data/dataset_path")),
    #"ICE_REFINED": (
     #   np.load("./data/dataset_path"),
      #  np.load("./data/dataset_path")),
    #"INCLINE_GRASS": (
     #   np.load("./data/dataset_path"),
      #  np.load("./data/dataset_path")),
    ##   np.load("./data/dataset_path"),
      #  np.load("./data/dataset_path")),
    "FLYING_TROT_MUD": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "CONCRETE": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "ASPHALT_REFINED": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "TROT_MOUNTEDPLATES_BRICKS": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "TROT_MOUNTEDPLATES_BRICKS_EXTREME": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "MUDTROT": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
   
    "WAREHOUSEASPHALTTROT": ( 
       np.load("./data/dataset_path"), 
       np.load("./data/dataset_path")
    ),
    
    "ASPHALT": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path")),
    "DRIVINGTERRAIN": (
        np.load("./data/dataset_path"),
        np.load("./data/dataset_path"))
}


def time_stats(data, time_idx=-1):
    t = data[..., time_idx]
    return {"mean": float(t.mean()), "min": float(t.min()), "max": float(t.max()), "std": float(t.std())}

# 1) Verify shapes and label lengths
# This loop should now run correctly!
for name, (data, labels) in terrains.items():
    print(name, "data:", data.shape, "labels:", labels.shape)
    if data.shape[0] != labels.shape[0]:
        print(" >>> MISMATCH lengths!", name)

# 2) Print time stats for all terrains
for name,(data,_) in terrains.items():
    stats = time_stats(data, time_idx=-1)
    print(f"{name}: mean={stats['mean']:.6f}, min={stats['min']:.6f}, max={stats['max']:.6f}")

# 3) Check class balance (if labels are binary or categorical)
for name,(_,labels) in terrains.items():
    vals,counts = np.unique(labels, return_counts=True)
    print(name, dict(zip(vals.tolist(), counts.tolist())))
    
for name, (data, _) in terrains.items():
    t = data[:, -1]
    print(f"{name}: mean={t.mean():.4f}, min={t.min():.4f}, max={t.max():.4f}")

for terrain_name, (data, labels) in terrains.items():
    assert data.shape[0] == labels.shape[0], f"Mismatch in {terrain_name}"

    n = data.shape[0]
    train_end = int(0.85 * n)

    X_train = data[:train_end]
    Y_train = labels[:train_end]
    X_val = data[train_end:]
    Y_val = labels[train_end:]

    np.save(os.path.join(save_dir, f"X_train_{terrain_name}.npy"), X_train)
    np.save(os.path.join(save_dir, f"Y_train_{terrain_name}.npy"), Y_train)
    np.save(os.path.join(save_dir, f"X_val_{terrain_name}.npy"), X_val)
    np.save(os.path.join(save_dir, f"Y_val_{terrain_name}.npy"), Y_val)

    # Note: print_time_summary_for_terrains should probably be called outside this loop 
    # if you only want to see the summary once, but it works here.
    print_time_summary_for_terrains(terrains, time_idx=-1)
    print(f"{terrain_name} Splits:")
    print(f"  Train -> X: {X_train.shape}, Y: {Y_train.shape}")
    print(f"  Val   -> X: {X_val.shape}, Y: {Y_val.shape}")
    print(f"  Time (train) range: [{X_train[..., -1].min():.6f}, {X_train[..., -1].max():.6f}]")
    print(f"  Time (val)   range: [{X_val[..., -1].min():.6f}, {X_val[..., -1].max():.6f}]")
    print("Saved splits successfully.\n")

In [ ]:
import os
import numpy as np

# ======================================================
#  GLOBAL STANDARDISATION (TRAIN MEAN / STD ONLY)
# ======================================================
def standardise(train_data, val_data):
    mean = train_data.mean(axis=0, keepdims=True)
    std  = train_data.std(axis=0, keepdims=True)
    std[std == 0] = 1e-8  # avoid division by zero
    return (train_data - mean) / std, (val_data - mean) / std


# ======================================================
#  BASE DIRECTORIES
# ======================================================
BASE_DIRS = [
    "./data/dataset_path",
    "./data/dataset_path"
]

TERRAIN_NAMES = [
    "2025-11-27-00-10-40",
    "2025-11-27-23-36-32",
    "2025-11-27-23-42-25",
    "2025-11-27-23-46-54",
    "2025-11-28-00-12-53",
    "2025-11-28-00-46-21",
    "2025-11-21-16-14-15",
    "2025-11-21-19-58-18",
    "2025-11-21-20-18-51",
    "2025-11-22-01-30-44",
    "2025-11-24-02-10-41",
    "2025-11-25-22-26-08",
    "2025-11-26-01-01-29",
    "2025-11-26-20-34-54",
    "concretetrot0.9",
    "2025-10-12-06-30-58mudflyingtrot",
    "2025-10-12-05-44-08sandpaperftterrain",
    "STANDING_TROT",
    "SANDPAPER",
    "MIDDLE_PEBBLE",
    "FLYING_TROT_MUD",
    "CONCRETE",
    "ASPHALT_REFINED",
    "TROT_MOUNTEDPLATES_BRICKS",
    "TROT_MOUNTEDPLATES_BRICKS_EXTREME",
    "MUDTROT",
    "WAREHOUSEASPHALTTROT", # This was the previously fixed entry
    "ASPHALT",
    "DRIVINGTERRAIN"
]



# ======================================================
#  HELPERS TO LOCATE FILES
# ======================================================
def find_file(terrain, prefix):
    for base in BASE_DIRS:
        p = os.path.join(base, "ALLSPLITTROT", f"{prefix}_{terrain}.npy")
        if os.path.exists(p):
            return p
    return None


def build_file_list(prefix):
    found, missing = [], []
    for terrain in TERRAIN_NAMES:
        f = find_file(terrain, prefix)
        if f:
            found.append(f)
        else:
            missing.append(terrain)

    print("\n======================================")
    print(f" FOUND {prefix} FILES:")
    for x in found:
        print("  ✓", x)

    print(f"\n MISSING {prefix} FILES:")
    for m in missing:
        print("  ✗", m)
    print("======================================\n")

    return found


# ======================================================
#  BUILD TRAIN/VAL FILESET LISTS
# ======================================================
X_train_files = build_file_list("X_train")
Y_train_files = build_file_list("Y_train")
X_val_files   = build_file_list("X_val")
Y_val_files   = build_file_list("Y_val")


# ======================================================
#  SAFE LOAD FUNCTION
# ======================================================
def load_many(files):
    arrays = []
    for f in files:
        if not os.path.exists(f):
            print(f"[WARNING] Missing file: {f}")
            continue
        arrays.append(np.load(f))
    if len(arrays) == 0:
        raise RuntimeError("No files loaded — all missing?")
    return np.concatenate(arrays, axis=0)


# ======================================================
#  LOAD ACTUAL DATA
# ======================================================
X_train = load_many(X_train_files)
Y_train = load_many(Y_train_files)
X_val   = load_many(X_val_files)
Y_val   = load_many(Y_val_files)

print("Raw concatenation complete:")
print("  X_train:", X_train.shape)
print("  Y_train:", Y_train.shape)
print("  X_val:",   X_val.shape)
print("  Y_val:",   Y_val.shape)


# ======================================================
#  GLOBAL STANDARDISATION
# ======================================================
X_train_std, X_val_std = standardise(X_train, X_val)


# ======================================================
#  SAVE FINAL SPLITS
# ======================================================
save_dir = "./data/dataset_path"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "X_train.npy"), X_train_std)
np.save(os.path.join(save_dir, "Y_train.npy"), Y_train)
np.save(os.path.join(save_dir, "X_val.npy"),   X_val_std)
np.save(os.path.join(save_dir, "Y_val.npy"),   Y_val)

print("\n============================================")
print("ALL SPLITS STANDARDISED + SAVED SUCCESSFULLY")
print("Saved to:", save_dir)
print("============================================\n")


In [ ]:
# === Global Standardization for Sim-to-Real Transfer ===
import os
import numpy as np

# CORRECTED: Use final_splitsTROT (with TROT suffix)
final_splits_dir = "./data/dataset_path"

# === 1. Load Concatenated Data ===
print("--- Loading concatenated data for Global Standardization ---")

# First, let's check what files actually exist
print(f"\nChecking directory: {final_splits_dir}")
if os.path.exists(final_splits_dir):
    files = os.listdir(final_splits_dir)
    print(f"Found {len(files)} files:")
    for f in sorted(files):
        if f.endswith('.npy'):
            size_mb = os.path.getsize(os.path.join(final_splits_dir, f)) / (1024*1024)
            print(f"  ✓ {f} ({size_mb:.2f} MB)")
else:
    print(f"ERROR: Directory does not exist!")
    exit(1)

# Load the files (they should be named train.npy and val.npy based on your output)
try:
    X_train = np.load(os.path.join(final_splits_dir, "train.npy"))
    X_val = np.load(os.path.join(final_splits_dir, "val.npy"))
    Y_train = np.load(os.path.join(final_splits_dir, "train_label.npy"))
    Y_val = np.load(os.path.join(final_splits_dir, "val_label.npy"))
except FileNotFoundError as e:
    print(f"\nERROR: Could not find files. Looking for:")
    print(f"  - train.npy")
    print(f"  - val.npy")
    print(f"  - train_label.npy")
    print(f"  - val_label.npy")
    print(f"\nActual files in directory:")
    for f in sorted(os.listdir(final_splits_dir)):
        print(f"  {f}")
    raise e

print(f"\nLoaded X_train shape: {X_train.shape}")
print(f"Loaded X_val shape: {X_val.shape}")
print(f"Loaded Y_train shape: {Y_train.shape}")
print(f"Loaded Y_val shape: {Y_val.shape}")

# === 2. Calculate Global Statistics (Mean and STD) from X_train ===
print("\n--- Calculating Global Mean and STD ---")

# Calculate mean and standard deviation across all samples (axis=0 = per-feature)
mu_global = X_train.mean(axis=0)
sigma_global = X_train.std(axis=0)

# CRITICAL: Handle features with zero standard deviation (constant features)
# This prevents division by zero errors during standardization
num_const_features = np.sum(sigma_global == 0)
if num_const_features > 0:
    print(f"WARNING: Found {num_const_features} constant features (std=0). Setting their std to 1.0")
    sigma_global[sigma_global == 0] = 1.0

# Save these stats for real-time inference (CRITICAL for Sim-to-Real)
np.save(os.path.join(final_splits_dir, "mu_global.npy"), mu_global)
np.save(os.path.join(final_splits_dir, "sigma_global.npy"), sigma_global)

print(f"\n✓ Global Stats saved to {final_splits_dir}")
print(f"  mu_global shape: {mu_global.shape}")
print(f"  sigma_global shape: {sigma_global.shape}")
print(f"  mu_global range: [{mu_global.min():.4f}, {mu_global.max():.4f}]")
print(f"  sigma_global range: [{sigma_global.min():.4f}, {sigma_global.max():.4f}]")

# === 3. Apply Global Standardization ===
print("\n--- Applying Global Standardization to Train and Val Splits ---")

# Apply Global Z-score Standardization: z = (x - μ) / σ
X_train_standardized = (X_train - mu_global) / sigma_global
X_val_standardized = (X_val - mu_global) / sigma_global

# Sanity check: Standardized training data should have mean≈0, std≈1
train_mean_after = X_train_standardized.mean(axis=0).mean()
train_std_after = X_train_standardized.std(axis=0).mean()
print(f"\nStandardization sanity check (training data):")
print(f"  Mean after standardization: {train_mean_after:.6f} (should be ≈0)")
print(f"  Std after standardization:  {train_std_after:.6f} (should be ≈1)")

# Check for NaN or Inf values (these indicate problems)
if np.isnan(X_train_standardized).any():
    print("\nERROR: NaN values detected in standardized training data!")
    nan_cols = np.where(np.isnan(X_train_standardized).any(axis=0))[0]
    print(f"  Columns with NaN: {nan_cols}")
    
if np.isinf(X_train_standardized).any():
    print("\nERROR: Inf values detected in standardized training data!")

# === 4. Save Standardized Data ===
print("\n--- Saving Standardized Data ---")

# Overwrite the original files with standardized versions
np.save(os.path.join(final_splits_dir, "train.npy"), X_train_standardized)
np.save(os.path.join(final_splits_dir, "val.npy"), X_val_standardized)

# Labels remain unchanged (no normalization needed)
# They should already exist, but we can verify
assert os.path.exists(os.path.join(final_splits_dir, "train_label.npy"))
assert os.path.exists(os.path.join(final_splits_dir, "val_label.npy"))

print(f"\n✓ Saved final, globally standardized data to: {final_splits_dir}")
print(f"  ✓ train.npy: {X_train_standardized.shape}")
print(f"  ✓ val.npy: {X_val_standardized.shape}")
print(f"  ✓ train_label.npy: {Y_train.shape}")
print(f"  ✓ val_label.npy: {Y_val.shape}")
print(f"  ✓ mu_global.npy: {mu_global.shape}")
print(f"  ✓ sigma_global.npy: {sigma_global.shape}")

print("\n" + "="*60)
print(" Data is now ready for stable training and Sim-to-Real deployment!")
print("="*60)
print("\nNext steps:")
print("1. Update your config YAML:")
print(f'   data_folder: "{final_splits_dir}"')
print("2. Run training:")
print("   python src/train_lightning.py --config your_config.yaml")




In [ ]:
import os, re, json
import numpy as np

# 12 canonical terrains
terrain_keys = [
    "asphalt", "plates", "warehouse", "concrete", "bricks",
    "gravel", "vcrterrain", "grass", "pebbles",
    "sandpaper", "ice", "mud"
]
TERRAIN_TO_ID = {k: i for i, k in enumerate(terrain_keys)}

# Aliases/synonyms seen in your filenames → canonical terrain
ALIASES = {
    "asphalt": [
        "asphalt", "asphaltpace", "asphaltpacetd",
        "asphaltskippinggait", "warehouseasphalttrot"
    ],
    "plates": [
        "plates", "mountedplates", "trot_mountedplates", "staticwalkmountedplates"
    ],
    "warehouse": [
        "warehouse", "warehousepawup", "warehouseasphalttrot"
    ],
    "concrete": ["concrete"],
    "bricks": ["bricks", "dynamic_walk_bricks"],
    "gravel": ["gravel", "gravel_incline"],
    "vcrterrain": ["vcrterrain", "skipvrcterrain", "vrc"],
    "grass": ["grass", "incline_grass"],
    "pebbles": ["pebbles", "middle_pebble", "pebble"],
    "sandpaper": ["sandpaper", "sandpaperftterrain"],
    "ice": ["ice", "ice_refined"],
    "mud": ["mud", "mudtrot", "flying_trot_mud", "flyingtrot_mud", "mudflyingtrot"]
}

# Build a quick reverse lookup of substring -> canonical terrain
SUBSTRING_TO_TERRAIN = {}
for canon, subs in ALIASES.items():
    for s in subs:
        SUBSTRING_TO_TERRAIN[s.lower()] = canon

def infer_terrain_from_filename(path: str) -> str:
    """Return canonical terrain from filename using substring aliases."""
    name = os.path.basename(path).lower()
    # normalize a bit
    name = name.replace("-", "_")
    # try all substrings
    for sub, terrain in SUBSTRING_TO_TERRAIN.items():
        if sub in name:
            return terrain
    # last resort: try the canonical keys themselves
    for k in TERRAIN_TO_ID:
        if k in name:
            return k
    raise ValueError(f"Could not map filename to a terrain token: {path}")

def concat_with_terrain(x_files, y_files, save_dir, split_name):
    """Load, concat and save X/Y + terrain ids; also return arrays."""
    assert len(x_files) == len(y_files), "X and Y file lists must be same length"
    X_parts, Y_parts, T_parts = [], [], []
    unmatched = []

    for xf, yf in zip(x_files, y_files):
        x = np.load(xf)
        y = np.load(yf)
        if x.shape[0] != y.shape[0]:
            raise ValueError(f"Length mismatch: {xf} has {x.shape[0]} samples, {yf} has {y.shape[0]}")

        try:
            terrain_key = infer_terrain_from_filename(xf)
        except ValueError:
            unmatched.append(os.path.basename(xf))
            continue

        tid = TERRAIN_TO_ID[terrain_key]
        t = np.full((x.shape[0],), tid, dtype=np.int64)

        X_parts.append(x); Y_parts.append(y); T_parts.append(t)
        print(f"[{split_name}] {terrain_key:10s} (id={tid})  +{x.shape[0]}")

    if unmatched:
        print("\nWARNING: These files didn’t match any terrain alias:")
        for u in unmatched:
            print("  -", u)
        print("→ Add a substring in ALIASES for each of those filenames.")

    X = np.concatenate(X_parts, axis=0)
    Y = np.concatenate(Y_parts, axis=0)
    T = np.concatenate(T_parts, axis=0)

    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, f"X_{split_name}.npy"), X)
    np.save(os.path.join(save_dir, f"Y_{split_name}.npy"), Y)
    np.save(os.path.join(save_dir, f"terrain_{split_name}.npy"), T)

    # Save mapping once
    mapping_path = os.path.join(save_dir, "terrain_mapping.json")
    if not os.path.exists(mapping_path):
        with open(mapping_path, "w") as f:
            json.dump(TERRAIN_TO_ID, f, indent=2)

    print(f"[{split_name}] Saved: X_{split_name}.npy, Y_{split_name}.npy, terrain_{split_name}.npy")
    print(f"[{split_name}] Shapes: X={X.shape}, Y={Y.shape}, terrain={T.shape}")
    return X, Y, T


In [ ]:
import os
import numpy as np

root_dir = "./data/dataset_path"
terrains = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]

for terrain in terrains:
    terrain_dir = os.path.join(root_dir, terrain)
    try:
        x_train = np.load(os.path.join(terrain_dir, "x_train.npy"))
        y_train = np.load(os.path.join(terrain_dir, "y_train.npy"))
        x_val = np.load(os.path.join(terrain_dir, "x_val.npy"))
        y_val = np.load(os.path.join(terrain_dir, "y_val.npy"))
        x_test = np.load(os.path.join(terrain_dir, "x_test.npy"))
        y_test = np.load(os.path.join(terrain_dir, "y_test.npy"))

        assert x_train.shape[0] == y_train.shape[0], f"{terrain}: x_train and y_train length mismatch"
        assert x_val.shape[0] == y_val.shape[0], f"{terrain}: x_val and y_val length mismatch"
        assert x_test.shape[0] == y_test.shape[0], f"{terrain}: x_test and y_test length mismatch"

        print(f"{terrain}: All splits match in length. (train: {x_train.shape[0]}, val: {x_val.shape[0]}, test: {x_test.shape[0]})")
    except Exception as e:
        print(f"{terrain}: Error - {e}")

In [ ]:
##### CONCATENATE ALL TERRAIN DATASETS INTO SINGLE FILES #####

In [ ]:
# concatenating dataset 
import os
import numpy as np  

#----------------- CONFIG ------------------
input_root = "Home/Desktop/simdata/noimu/preprocessed_data"
output_file = "final_dataset.npy"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Helper function to load .npy files
def load_arrays(filenames):
    return [np.load(f) for f in filenames]

# X TRAIN
X_train_files = [
    "X_train_asphalt.npy",
    "X_train_concrete.npy",
    "X_train_gravel_incline.npy",
    "X_train_ice_refined.npy",
    "X_train_incline_grass.npy",
    "X_train_lindyhop_road.npy",
    "X_train_middle_pebble.npy",
    "X_train_sandpaper_refined.npy",
    "X_train_standing_trot_bricks.npy",
    "X_train_trot_mountedplates_bricks.npy",
    "X_train_flying_trot_mud.npy"
]
X_train = np.concatenate(load_arrays(X_train_files), axis=0)

# X VAL
X_val_files = [
    "X_val_asphalt.npy",
    "X_val_concrete.npy",
    "X_val_gravel_incline.npy",
    "X_val_ice_refined.npy",
    "X_val_incline_grass.npy",
    "X_val_lindyhop_road.npy",
    "X_val_middle_pebble.npy",
    "X_val_sandpaper_refined.npy",
    "X_val_standing_trot_bricks.npy",
    "X_val_trot_mountedplates_bricks.npy",
    "X_val_flying_trot_mud.npy"
]
X_val = np.concatenate(load_arrays(X_val_files), axis=0)

# X TEST
X_test_files = [
    "X_test_asphalt.npy",
    "X_test_concrete.npy",
    "X_test_gravel_incline.npy",
    "X_test_ice_refined.npy",
    "X_test_incline_grass.npy",
    "X_test_lindyhop_road.npy",
    "X_test_middle_pebble.npy",
    "X_test_sandpaper_refined.npy",
    "X_test_standing_trot_bricks.npy",
    "X_test_trot_mountedplates_bricks.npy",
    "X_test_flying_trot_mud.npy"
]
X_test = np.concatenate(load_arrays(X_test_files), axis=0)

# Y TRAIN
Y_train_files = [
    "Y_train_asphalt.npy",
    "Y_train_concrete.npy",
    "Y_train_gravel_incline.npy",
    "Y_train_ice_refined.npy",
    "Y_train_incline_grass.npy",
    "Y_train_lindyhop_road.npy",
    "Y_train_middle_pebble.npy",
    "Y_train_sandpaper_refined.npy",
    "Y_train_standing_trot_bricks.npy",
    "Y_train_trot_mountedplates_bricks.npy",
    "Y_train_flying_trot_mud.npy"
]
Y_train = np.concatenate(load_arrays(Y_train_files), axis=0)

# Y VAL
Y_val_files = [
    "Y_val_asphalt.npy",
    "Y_val_concrete.npy",
    "Y_val_gravel_incline.npy",
    "Y_val_ice_refined.npy",
    "Y_val_incline_grass.npy",
    "Y_val_lindyhop_road.npy",
    "Y_val_middle_pebble.npy",
    "Y_val_sandpaper_refined.npy",
    "Y_val_standing_trot_bricks.npy",
    "Y_val_trot_mountedplates_bricks.npy",
    "Y_val_flying_trot_mud.npy"
]
Y_val = np.concatenate(load_arrays(Y_val_files), axis=0)

# Y TEST
Y_test_files = [
    "Y_test_asphalt.npy",
    "Y_test_concrete.npy",
    "Y_test_gravel_incline.npy",
    "Y_test_ice_refined.npy",
    "Y_test_incline_grass.npy",
    "Y_test_lindyhop_road.npy",
    "Y_test_middle_pebble.npy",
    "Y_test_sandpaper_refined.npy",
    "Y_test_standing_trot_bricks.npy",
    "Y_test_trot_mountedplates_bricks.npy",
    "Y_test_flying_trot_mud.npy"
]
Y_test = np.concatenate(load_arrays(Y_test_files), axis=0)
